# NBA Advanced Stats Scraper

Scrapes BoxScoreAdvancedV3 data for all games in the database.

**Features:**
- Resumable: Tracks progress in `scrape_advanced_stats_progress` table
- Crash-safe: Commits every 50 games
- Rate-limited: Random delays + long pauses to avoid API bans
- DNP handling: Flags players who didn't play

**Before running:**
1. Execute `advanced_stats_schema.sql` in Supabase to create tables
2. Ensure `.env` file has `DATABASE_URL`

In [77]:
# =============================================================================
# Imports and Setup
# =============================================================================

from requests.exceptions import ReadTimeout, ConnectionError
from sqlalchemy import create_engine, text, inspect
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import numpy as np
import random
import time
import json
import re
import os

from nba_api.stats.endpoints import boxscoreadvancedv3

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Imports complete")

Imports complete


In [78]:
# =============================================================================
# Configuration
# =============================================================================

# Scraper settings
BATCH_SIZE = 50                    # Commit to DB every N games
SHORT_DELAY_MIN = 0.1              # Min seconds between API calls
SHORT_DELAY_MAX = 1.5              # Max seconds between API calls  
LONG_PAUSE_EVERY = 100             # Long pause every N games
LONG_PAUSE_MIN = 30                # Min seconds for long pause
LONG_PAUSE_MAX = 100               # Max seconds for long pause
ERROR_RETRY_DELAY = 120            # Seconds to wait after error
MAX_RETRIES = 3                    # Max retries per game before skipping

# Database connection
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment. Check your .env file.")

engine = create_engine(DATABASE_URL)
print(f"Connected to database")

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Database connection verified ✓")

Connected to database
Database connection verified ✓


In [82]:
# =============================================================================
# Helper Functions
# =============================================================================

def camel_to_snake(name: str) -> str:
    """
    Convert camelCase to snake_case.
    e.g., 'offensiveRating' -> 'offensive_rating'
          'PIE' -> 'pie'
    """
    # Handle acronyms like PIE
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()


def parse_minutes(minutes_str) -> float:
    """
    Parse minutes from various formats to float.
    
    Handles:
    - 'MM:SS' format (e.g., '28:17' -> 28.283)
    - 'PT28M17.00S' ISO format
    - Already numeric values
    - None/empty/NaN -> 0.0
    """
    if pd.isna(minutes_str) or minutes_str == '' or minutes_str is None:
        return 0.0
    
    # Already numeric
    if isinstance(minutes_str, (int, float)):
        return float(minutes_str)
    
    minutes_str = str(minutes_str)
    
    # MM:SS format (BoxScoreAdvancedV3)
    if ':' in minutes_str and 'PT' not in minutes_str:
        try:
            parts = minutes_str.split(':')
            mins = int(parts[0])
            secs = int(parts[1]) if len(parts) > 1 else 0
            return round(mins + secs / 60, 3)
        except (ValueError, IndexError):
            return 0.0
    
    # ISO 8601 format (PT28M17.00S)
    if 'PT' in minutes_str:
        try:
            match = re.match(r'PT(\d+)M([\d.]+)S', minutes_str)
            if match:
                mins = int(match.group(1))
                secs = float(match.group(2))
                return round(mins + secs / 60, 3)
        except (ValueError, AttributeError):
            return 0.0
    
    return 0.0


def determine_dnp(row: pd.Series) -> bool:
    """
    Determine if a player did not play.
    
    DNP if:
    - 'comment' field contains text (usually DNP reason)
    - minutes is 0 or null
    """
    comment = row.get('comment', '')
    minutes = row.get('minutes', 0)
    
    # Has a comment (DNP reason)
    if pd.notna(comment) and str(comment).strip() != '':
        return True
    
    # Zero or null minutes
    if pd.isna(minutes) or minutes == 0:
        return True
    
    return False


def transform_player_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform player DataFrame from API format to database format.
    """
    if df.empty:
        return df
    
    # Rename columns: camelCase -> snake_case
    df = df.rename(columns={col: camel_to_snake(col) for col in df.columns})
    
    # Parse minutes
    if 'minutes' in df.columns:
        df['minutes'] = df['minutes'].apply(parse_minutes)
    
    # Add DNP flag
    df['did_not_play'] = df.apply(determine_dnp, axis=1)
    
    # Add timestamp
    df['created_at'] = datetime.now()
    df = df.drop_duplicates(subset=['game_id', 'person_id'], keep='first')
    return df


def transform_team_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform team DataFrame from API format to database format.
    """
    if df.empty:
        return df
    
    # Rename columns: camelCase -> snake_case
    df = df.rename(columns={col: camel_to_snake(col) for col in df.columns})
    
    # Parse minutes
    if 'minutes' in df.columns:
        df['minutes'] = df['minutes'].apply(parse_minutes)
    
    # Add timestamp
    df['created_at'] = datetime.now()

    return df


def rate_limit(game_num: int):
    """
    Apply rate limiting between API calls.
    Long pause every LONG_PAUSE_EVERY games.
    """
    if game_num > 0 and game_num % LONG_PAUSE_EVERY == 0:
        pause = round(random.uniform(LONG_PAUSE_MIN, LONG_PAUSE_MAX), 1)
        print(f"\n  ⏸️  Long pause: {pause}s (every {LONG_PAUSE_EVERY} games)\n")
        time.sleep(pause)
    else:
        delay = round(random.uniform(SHORT_DELAY_MIN, SHORT_DELAY_MAX), 2)
        time.sleep(delay)


print("Helper functions loaded ✓")

# Test camel_to_snake
test_cases = ['offensiveRating', 'PIE', 'assistToTurnover', 'gameId', 'estimatedOffensiveRating']
print("\nColumn name transformation test:")
for tc in test_cases:
    print(f"  {tc} -> {camel_to_snake(tc)}")

Helper functions loaded ✓

Column name transformation test:
  offensiveRating -> offensive_rating
  PIE -> pie
  assistToTurnover -> assist_to_turnover
  gameId -> game_id
  estimatedOffensiveRating -> estimated_offensive_rating


In [83]:
# =============================================================================
# 1. TRUTH-BASED GAME FETCHING
# =============================================================================

def get_actual_missing_game_ids(engine):
    """
    Compares the full schedule (team_game_stats) against the 
    actual saved data (advanced_player_game_stats).
    Returns ONLY the game_ids that are missing.
    """
    sql = """
    SELECT DISTINCT t.game_id
    FROM team_game_stats t
    LEFT JOIN advanced_player_game_stats a ON t.game_id = a.game_id
    WHERE a.game_id IS NULL
    ORDER BY t.game_id ASC;
    """
    
    print("Calculating missing games based on actual table data...")
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        missing_ids = [row[0] for row in result]
        
    return missing_ids

# Load the list
games_to_scrape = get_actual_missing_game_ids(engine)

print(f"{'='*40}")
print(f"ACTUAL MISSING GAMES: {len(games_to_scrape)}")
print(f"{'='*40}")
# Preview first 5 to make sure they look like valid IDs
print(f"Next 5 games: {games_to_scrape[:5]}")

Calculating missing games based on actual table data...
ACTUAL MISSING GAMES: 1969
Next 5 games: ['0021700823', '0021800012', '0021800033', '0021800051', '0021800066']


In [84]:
# =============================================================================
# Single Game Scraper (for testing)
# =============================================================================

def scrape_single_game(game_id: str) -> tuple:
    """
    Scrape advanced stats for a single game.
    
    Returns:
        (player_df, team_df) - Both transformed and ready for DB insertion
    """
    box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=game_id)
    dfs = box_adv.get_data_frames()
    
    # dfs[0] = players, dfs[1] = teams
    player_df = transform_player_df(dfs[0].copy())
    team_df = transform_team_df(dfs[1].copy())
    
    return player_df, team_df


# Test with first game
if games_to_scrape:
    test_game_id = games_to_scrape[0]
    print(f"Testing with game: {test_game_id}")
    
    try:
        player_df, team_df = scrape_single_game(test_game_id)
        print(f"\nPlayer DataFrame:")
        print(f"  Shape: {player_df.shape}")
        print(f"  Columns: {list(player_df.columns)}")
        print(f"  DNP players: {player_df['did_not_play'].sum()}")
        
        print(f"\nTeam DataFrame:")
        print(f"  Shape: {team_df.shape}")
        print(f"  Columns: {list(team_df.columns)}")
        
        print("\n✓ Test successful")
    except Exception as e:
        print(f"\n❌ Test failed: {e}")
else:
    print("No games to scrape - all done!")

Testing with game: 0021700823

Player DataFrame:
  Shape: (25, 39)
  Columns: ['game_id', 'team_id', 'team_city', 'team_name', 'team_tricode', 'team_slug', 'person_id', 'first_name', 'family_name', 'name_i', 'player_slug', 'position', 'comment', 'jersey_num', 'minutes', 'estimated_offensive_rating', 'offensive_rating', 'estimated_defensive_rating', 'defensive_rating', 'estimated_net_rating', 'net_rating', 'assist_percentage', 'assist_to_turnover', 'assist_ratio', 'offensive_rebound_percentage', 'defensive_rebound_percentage', 'rebound_percentage', 'turnover_ratio', 'effective_field_goal_percentage', 'true_shooting_percentage', 'usage_percentage', 'estimated_usage_percentage', 'estimated_pace', 'pace', 'pace_per40', 'possessions', 'pie', 'did_not_play', 'created_at']
  DNP players: 5

Team DataFrame:
  Shape: (2, 31)
  Columns: ['game_id', 'team_id', 'team_city', 'team_name', 'team_tricode', 'team_slug', 'minutes', 'estimated_offensive_rating', 'offensive_rating', 'estimated_defensive_r

In [85]:
# =============================================================================
# 2. DIRECT SCRAPER (NO PROGRESS TABLE)
# =============================================================================

import time
import random
from datetime import datetime
from requests.exceptions import ReadTimeout, ConnectionError, ChunkedEncodingError

# Settings
MAX_RETRIES = 3
BAN_COOLDOWN = 600  # 10 Minutes sleep if we get a Timeout/Connection Refused

total = len(games_to_scrape)
print(f"Starting scrape for {total} games...")

for i, game_id in enumerate(games_to_scrape, 1):
    
    success = False
    attempt = 0
    
    while attempt < MAX_RETRIES and not success:
        attempt += 1
        try:
            print(f"[{i}/{total}] Scraping {game_id} (Try {attempt})... ", end="", flush=True)
            
            # 1. Scrape
            player_df, team_df = scrape_single_game(game_id)
            
            dupes = player_df[player_df.duplicated(subset=['game_id', 'person_id'], keep=False)]
            if not dupes.empty:
                print(f"Found {len(dupes)} duplicate rows:")
                print(dupes[['game_id', 'person_id', 'first_name', 'family_name', 'team_tricode']])
                
            # 2. Check if data is empty (sometimes API returns blank for old/preseason games)
            if player_df.empty and team_df.empty:
                print("⚠️  Empty data returned (Skipping)")
                # We stop retrying; if NBA has no data, they have no data.
                break 

            # 3. Direct Save to DB (Transaction)
            with engine.begin() as conn:
                if not player_df.empty:
                    player_df.to_sql('advanced_player_game_stats', conn, if_exists='append', index=False)
                if not team_df.empty:
                    team_df.to_sql('advanced_team_game_stats', conn, if_exists='append', index=False)
            
            print("✓ Saved")
            success = True
            
            # 4. Standard Rate Limit (Short delay)
            time.sleep(random.uniform(0.6, 1.2))

        except (ReadTimeout, ConnectionError, ConnectionResetError) as e:
            # THIS HANDLES THE IP BAN
            print(f"\n🛑 CONNECTION BLOCKED: {e}")
            print(f"💤 Sleeping for {BAN_COOLDOWN/60} minutes to reset connection...")
            time.sleep(BAN_COOLDOWN)
            # Loop will continue and retry the same game after waking up
            
        except Exception as e:
            print(f"\n❌ Error: {e}")
            if "duplicate key" in str(e) or "UniqueViolation" in str(e):
                print("   (Game already exists, moving on)")
                success = True # Treat as success so we move to next game
            else:
                time.sleep(5) # Short sleep for random logic errors

    if not success:
        print(f"💀 Giving up on {game_id} after {MAX_RETRIES} attempts.")

print("\nJob Complete.")

Starting scrape for 1969 games...
[1/1969] Scraping 0021700823 (Try 1)... ✓ Saved
[2/1969] Scraping 0021800012 (Try 1)... ✓ Saved
[3/1969] Scraping 0021800033 (Try 1)... ✓ Saved
[4/1969] Scraping 0021800051 (Try 1)... ✓ Saved
[5/1969] Scraping 0021800066 (Try 1)... ✓ Saved
[6/1969] Scraping 0021800145 (Try 1)... ✓ Saved
[7/1969] Scraping 0021800171 (Try 1)... ✓ Saved
[8/1969] Scraping 0021800187 (Try 1)... ✓ Saved
[9/1969] Scraping 0021800214 (Try 1)... ✓ Saved
[10/1969] Scraping 0021800228 (Try 1)... ✓ Saved
[11/1969] Scraping 0021800276 (Try 1)... ✓ Saved
[12/1969] Scraping 0021800282 (Try 1)... ✓ Saved
[13/1969] Scraping 0021800304 (Try 1)... ✓ Saved
[14/1969] Scraping 0021800329 (Try 1)... ✓ Saved
[15/1969] Scraping 0021800347 (Try 1)... ✓ Saved
[16/1969] Scraping 0021800356 (Try 1)... ✓ Saved
[17/1969] Scraping 0021800368 (Try 1)... ✓ Saved
[18/1969] Scraping 0021800380 (Try 1)... ✓ Saved
[19/1969] Scraping 0021800398 (Try 1)... ✓ Saved
[20/1969] Scraping 0021800426 (Try 1)... ✓ S

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Verification Queries
# =============================================================================

print("Checking scrape results...\n")

# Row counts
player_count = pd.read_sql("SELECT COUNT(*) as cnt FROM advanced_player_game_stats", engine)['cnt'].iloc[0]
team_count = pd.read_sql("SELECT COUNT(*) as cnt FROM advanced_team_game_stats", engine)['cnt'].iloc[0]
progress_count = pd.read_sql("SELECT COUNT(*) as cnt FROM scrape_advanced_stats_progress WHERE status = 'success'", engine)['cnt'].iloc[0]

print(f"Player stats rows: {player_count:,}")
print(f"Team stats rows: {team_count:,}")
print(f"Games successfully scraped: {progress_count:,}")

# Failed games
failed_games = pd.read_sql("""
    SELECT game_id, scraped_at 
    FROM scrape_advanced_stats_progress 
    WHERE status = 'failed'
    ORDER BY scraped_at DESC
""", engine)

if len(failed_games) > 0:
    print(f"\n⚠️  Failed games: {len(failed_games)}")
    print(failed_games.head(10))
else:
    print(f"\n✅ No failed games")

# DNP distribution
dnp_dist = pd.read_sql("""
    SELECT did_not_play, COUNT(*) as count
    FROM advanced_player_game_stats
    GROUP BY did_not_play
""", engine)
print(f"\nDNP distribution:")
print(dnp_dist)

In [ ]:
# =============================================================================
# Retry Failed Games (Optional)
# =============================================================================

# Get failed games
failed_games = pd.read_sql("""
    SELECT game_id 
    FROM scrape_advanced_stats_progress 
    WHERE status = 'failed'
""", engine)

if len(failed_games) > 0:
    failed_game_ids = failed_games['game_id'].tolist()
    print(f"Found {len(failed_game_ids)} failed games to retry")
    
    # First, delete the failed progress records so they can be re-inserted
    # Uncomment to retry:
    # with engine.connect() as conn:
    #     conn.execute(text("DELETE FROM scrape_advanced_stats_progress WHERE status = 'failed'"))
    #     conn.commit()
    # 
    # processed, failed = run_scraper(failed_game_ids, engine)
else:
    print("No failed games to retry")

In [ ]:
# =============================================================================
# Sample Data Preview
# =============================================================================

print("Sample player advanced stats:")
sample_player = pd.read_sql("""
    SELECT game_id, person_id, first_name, family_name, team_tricode,
           minutes, offensive_rating, defensive_rating, net_rating,
           usage_percentage, true_shooting_percentage, pie, did_not_play
    FROM advanced_player_game_stats
    WHERE did_not_play = FALSE
    LIMIT 10
""", engine)
display(sample_player)

print("\nSample team advanced stats:")
sample_team = pd.read_sql("""
    SELECT game_id, team_id, team_tricode,
           minutes, offensive_rating, defensive_rating, net_rating,
           pace, possessions, pie
    FROM advanced_team_game_stats
    LIMIT 10
""", engine)
display(sample_team)